# Observabilidad en Aplicaciones LLM

> Aprende a instrumentar tus llamadas a LLMs para registrar latencia, coste y uso de tokens.
> Analizaremos los logs con pandas y visualizaremos el coste acumulado con matplotlib.

## Objetivos
- Implementar un decorador de logging para llamadas a la API
- Capturar métricas: latencia, tokens de entrada/salida, coste estimado
- Analizar los logs con pandas
- Visualizar el coste acumulado a lo largo del tiempo

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pandas matplotlib --quiet

## 2. Configuración e infraestructura de logging

In [ ]:
import anthropic
import time
import json
import functools
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from typing import Callable

# Precios por millón de tokens (claude-haiku-4-5, USD)
PRECIO_INPUT_POR_MTOKEN = 0.80
PRECIO_OUTPUT_POR_MTOKEN = 4.00

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Almacén en memoria para los logs de esta sesión
registro_llamadas = []

print("Configuración lista.")
print(f"Precios: ${PRECIO_INPUT_POR_MTOKEN}/M tokens entrada | ${PRECIO_OUTPUT_POR_MTOKEN}/M tokens salida")

## 3. Decorador de logging

Creamos un decorador que envuelve cualquier llamada a la API de Anthropic
y registra automáticamente las métricas relevantes.

In [ ]:
def log_llamada_llm(func: Callable) -> Callable:
    """Decorador que registra métricas de cada llamada al LLM."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        inicio = time.time()
        timestamp = datetime.now()
        error = None
        respuesta = None

        try:
            respuesta = func(*args, **kwargs)
        except Exception as e:
            error = str(e)
            raise
        finally:
            latencia_ms = (time.time() - inicio) * 1000

            # Extraer métricas de uso si la llamada fue exitosa
            if respuesta is not None:
                tokens_entrada = respuesta.usage.input_tokens
                tokens_salida = respuesta.usage.output_tokens
                coste_usd = (
                    tokens_entrada / 1_000_000 * PRECIO_INPUT_POR_MTOKEN +
                    tokens_salida / 1_000_000 * PRECIO_OUTPUT_POR_MTOKEN
                )
                modelo = respuesta.model
            else:
                tokens_entrada = tokens_salida = coste_usd = 0
                modelo = MODELO

            entrada = kwargs.get("messages", args[0] if args else [{}])
            nombre_operacion = kwargs.get("_operacion", func.__name__)

            entrada_preview = ""
            if isinstance(entrada, list) and entrada:
                contenido = entrada[-1].get("content", "")
                if isinstance(contenido, str):
                    entrada_preview = contenido[:80]

            registro = {
                "timestamp": timestamp,
                "operacion": nombre_operacion,
                "modelo": modelo,
                "latencia_ms": round(latencia_ms, 1),
                "tokens_entrada": tokens_entrada,
                "tokens_salida": tokens_salida,
                "tokens_total": tokens_entrada + tokens_salida,
                "coste_usd": round(coste_usd, 6),
                "error": error,
                "entrada_preview": entrada_preview
            }
            registro_llamadas.append(registro)

            estado = "OK" if error is None else "ERROR"
            print(f"[{estado}] {nombre_operacion:20s} | {latencia_ms:7.0f}ms | "
                  f"{tokens_entrada}+{tokens_salida} tokens | ${coste_usd:.5f}")

        return respuesta
    return wrapper


@log_llamada_llm
def llamar_api(messages: list, max_tokens: int = 256, **kwargs) -> anthropic.types.Message:
    """Función base para llamadas a la API con logging automático."""
    return client.messages.create(
        model=MODELO,
        max_tokens=max_tokens,
        messages=messages
    )

print("Decorador de logging listo.")

## 4. Llamadas de prueba con logging

Realizamos varias llamadas simulando diferentes operaciones de una aplicación real.

In [ ]:
# Simulamos diferentes tipos de operaciones en una aplicación
operaciones = [
    {
        "_operacion": "resumen_texto",
        "messages": [{"role": "user",
            "content": "Resume en 2 frases: La inteligencia artificial es una rama de la informática que busca crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana."}],
        "max_tokens": 128
    },
    {
        "_operacion": "clasificacion",
        "messages": [{"role": "user",
            "content": "Clasifica esta reseña como POSITIVA, NEGATIVA o NEUTRAL: 'El producto llegó en buen estado pero el envío tardó más de lo esperado.'"}],
        "max_tokens": 64
    },
    {
        "_operacion": "generacion_respuesta",
        "messages": [{"role": "user",
            "content": "¿Cuáles son las ventajas del aprendizaje por refuerzo comparado con el aprendizaje supervisado?"}],
        "max_tokens": 256
    },
    {
        "_operacion": "extraccion_entidades",
        "messages": [{"role": "user",
            "content": "Extrae personas, lugares y fechas de: 'El 15 de marzo de 2024, María García presentó su tesis en la Universidad de Madrid ante el profesor Juan López.'"}],
        "max_tokens": 128
    },
    {
        "_operacion": "traduccion",
        "messages": [{"role": "user",
            "content": "Traduce al inglés: 'El aprendizaje automático ha transformado la manera en que interactuamos con la tecnología cotidiana.'"}],
        "max_tokens": 128
    }
]

print("Realizando llamadas de prueba...")
print("-" * 75)
print(f"{'Estado':6} {'Operación':22} {'Latencia':>10} {'Tokens':>12} {'Coste':>10}")
print("-" * 75)

respuestas = []
for op in operaciones:
    _op = op.pop("_operacion")
    r = llamar_api(_operacion=_op, **op)
    respuestas.append(r)

print("-" * 75)
print(f"Total de llamadas registradas: {len(registro_llamadas)}")

## 5. Análisis de logs con pandas

In [ ]:
# Convertir los registros a DataFrame
df_logs = pd.DataFrame(registro_llamadas)
df_logs["timestamp"] = pd.to_datetime(df_logs["timestamp"])

print("=== ANÁLISIS DE LOGS ===")
print(f"\nTotal de llamadas: {len(df_logs)}")
print(f"Período: {df_logs['timestamp'].min().strftime('%H:%M:%S')} - {df_logs['timestamp'].max().strftime('%H:%M:%S')}")

print("\n--- Coste total ---")
coste_total = df_logs["coste_usd"].sum()
print(f"Coste total de la sesión: ${coste_total:.6f} USD")
print(f"Coste proyectado (1000 llamadas): ${coste_total / len(df_logs) * 1000:.4f} USD")

print("\n--- Latencia ---")
print(f"Latencia media: {df_logs['latencia_ms'].mean():.0f} ms")
print(f"Latencia mínima: {df_logs['latencia_ms'].min():.0f} ms")
print(f"Latencia máxima: {df_logs['latencia_ms'].max():.0f} ms")
print(f"Latencia p95: {df_logs['latencia_ms'].quantile(0.95):.0f} ms")

print("\n--- Distribución de tokens ---")
print(f"Tokens entrada (total): {df_logs['tokens_entrada'].sum():,}")
print(f"Tokens salida (total): {df_logs['tokens_salida'].sum():,}")
print(f"Tokens entrada (media por llamada): {df_logs['tokens_entrada'].mean():.0f}")
print(f"Tokens salida (media por llamada): {df_logs['tokens_salida'].mean():.0f}")

print("\n--- Por operación ---")
resumen = df_logs.groupby("operacion").agg(
    llamadas=("coste_usd", "count"),
    latencia_media=("latencia_ms", "mean"),
    tokens_media=("tokens_total", "mean"),
    coste_total=("coste_usd", "sum")
).round(2)
print(resumen.to_string())

## 6. Gráfica de coste acumulado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Dashboard de Observabilidad LLM", fontsize=14, fontweight="bold")

# 1. Coste acumulado
ax1 = axes[0, 0]
df_logs_sorted = df_logs.sort_values("timestamp")
df_logs_sorted["coste_acumulado"] = df_logs_sorted["coste_usd"].cumsum()
ax1.plot(range(1, len(df_logs_sorted)+1), df_logs_sorted["coste_acumulado"] * 1000,
         marker="o", linewidth=2, color="#e74c3c", markersize=6)
ax1.fill_between(range(1, len(df_logs_sorted)+1),
                 df_logs_sorted["coste_acumulado"] * 1000, alpha=0.2, color="#e74c3c")
ax1.set_title("Coste Acumulado de la Sesión")
ax1.set_xlabel("Número de llamada")
ax1.set_ylabel("Coste acumulado (millicents USD)")
ax1.grid(alpha=0.3)
ax1.set_xticks(range(1, len(df_logs_sorted)+1))

# 2. Latencia por operación
ax2 = axes[0, 1]
colores_op = ["#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c"]
bars = ax2.barh(df_logs["operacion"], df_logs["latencia_ms"],
                color=colores_op[:len(df_logs)], edgecolor="black", alpha=0.8)
ax2.set_title("Latencia por Operación")
ax2.set_xlabel("Latencia (ms)")
for bar, val in zip(bars, df_logs["latencia_ms"]):
    ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f"{val:.0f}ms", va="center", fontsize=9)
ax2.grid(axis="x", alpha=0.3)

# 3. Distribución de tokens
ax3 = axes[1, 0]
x = range(len(df_logs))
ancho = 0.35
ax3.bar([i - ancho/2 for i in x], df_logs["tokens_entrada"],
        ancho, label="Entrada", color="#3498db", alpha=0.8)
ax3.bar([i + ancho/2 for i in x], df_logs["tokens_salida"],
        ancho, label="Salida", color="#e74c3c", alpha=0.8)
ax3.set_title("Distribución de Tokens por Llamada")
ax3.set_xlabel("Llamada")
ax3.set_ylabel("Tokens")
ax3.set_xticks(x)
ax3.set_xticklabels(df_logs["operacion"], rotation=25, ha="right", fontsize=8)
ax3.legend()
ax3.grid(axis="y", alpha=0.3)

# 4. Coste por operación (pastel)
ax4 = axes[1, 1]
ax4.pie(df_logs["coste_usd"], labels=df_logs["operacion"],
        autopct="%1.1f%%", colors=colores_op[:len(df_logs)], startangle=90)
ax4.set_title("Distribución de Coste por Operación")

plt.tight_layout()
plt.savefig("observabilidad_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Dashboard guardado. Coste total de la sesión: ${coste_total:.6f} USD")